# Estrutura da Página HTML para RPA

## Link (GitHub Pages)

https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n

## Código HTML

```html
<!doctype html>
<html lang="pt-br">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Clientes</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 24px; }
    table { border-collapse: collapse; width: 100%; max-width: 900px; }
    th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
    th { background: #f5f5f5; }
  </style>
</head>
<body>
  <h1>Lista de Clientes</h1>

  <table id="clientes">
    <thead>
      <tr>
        <th>Nome</th>
        <th>Email</th>
        <th>Saldo</th>
        <th>Perfil</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td class="nome">Ana Silva</td>
        <td class="email">ana@email.com</td>
        <td class="saldo">R$ 12.500,00</td>
        <td class="perfil">Conservador</td>
      </tr>
      <tr>
        <td class="nome">Bruno Lima</td>
        <td class="email">bruno@email.com</td>
        <td class="saldo">R$ 3.200,00</td>
        <td class="perfil">Moderado</td>
      </tr>
      <tr>
        <td class="nome">Carla Souza</td>
        <td class="email">carla@email.com</td>
        <td class="saldo">R$ 28.900,00</td>
        <td class="perfil">Arrojado</td>
      </tr>
    </tbody>
  </table>
</body>
</html>
```

# Dependências (Python)

In [ ]:
!pip install requests beautifulsoup4

# Código-fonte RPA (Pyhton)

## Execução do RPA e envio ao n8n
Configure a URL de teste do Webhook antes de executar. Os dados são fictícios.

In [ ]:
import requests
from bs4 import BeautifulSoup

# CONFIGURAÇÕES
URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"
# Copie a URL de TESTE mostrada no nó Webhook do n8n após clicar em "Listen for test event".
N8N_WEBHOOK = "https://SEU-N8N.app.n8n.cloud/webhook-test/Clientes"

# 1. EXTRAIR CLIENTES
resposta = requests.get(URL_PAGINA, timeout=30)
resposta.raise_for_status()

soup = BeautifulSoup(resposta.text, "html.parser")
linhas = soup.select("#clientes tbody tr")
if not linhas:
    raise ValueError("Nenhum cliente encontrado na página.")

clientes = []
for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) < 4:
        raise ValueError("Linha da tabela incompleta.")
    saldo_texto = colunas[2].get_text(strip=True)
    saldo = float(
        saldo_texto.replace("R$", "").replace(".", "").replace(",", ".").strip()
    )
    cliente = {
        "nome": colunas[0].get_text(strip=True),
        "email": colunas[1].get_text(strip=True),
        "saldo": saldo,
        "perfil": colunas[3].get_text(strip=True)
    }
    clientes.append(cliente)

# 2. EXIBIR CLIENTES
print("CLIENTES EXTRAÍDOS")
print("=" * 45)
for cliente in clientes:
    print(
        f"{cliente['nome']} | "
        f"{cliente['perfil']} | "
        f"R$ {cliente['saldo']:,.2f}"
    )
print("=" * 45)
print(f"Total de clientes: {len(clientes)}")

# 3. ENVIAR PARA O N8N
if "SEU-N8N" in N8N_WEBHOOK:
    raise ValueError("Configure N8N_WEBHOOK com a URL de teste do seu n8n.")

print("\nEnviando clientes ao n8n...")
payload = {"clientes": clientes}
resposta_n8n = requests.post(N8N_WEBHOOK, json=payload, timeout=120)
print("Status HTTP:", resposta_n8n.status_code)
resposta_n8n.raise_for_status()
try:
    print("Resposta n8n:", resposta_n8n.json())
except ValueError:
    print("Resposta n8n:", resposta_n8n.text)
print("\nProcessamento concluído.")


In [ ]:
# DIAGNÓSTICO HTTP - SEM NOVO ENVIO
resposta = globals().get("resposta_n8n")
if resposta is None:
    print("A variável resposta_n8n não foi encontrada.")
    print("Execute a célula principal antes do diagnóstico.")
else:
    print("========== DIAGNÓSTICO HTTP ==========")
    print("Status:", resposta.status_code)
    print("Content-Type:", resposta.headers.get("Content-Type"))
    print("Tamanho:", len(resposta.content), "bytes")
    print("Corpo bruto:", repr(resposta.text[:2000]))
    print("======================================")
